In [6]:
import os
import sys

from dotenv import load_dotenv

root_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))

print(f"Root directory: {root_dir}")
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

load_dotenv(os.path.join(root_dir, ".env"))

Root directory: /home/victor-wsl/programming/curso-ia


True

# Pipeline Completa

In [ ]:
# import torch

# print(f"PyTorch version: {torch.__version__}")
# print(f"CUDA available: {torch.cuda.is_available()}")
# print(f"CUDA version: {torch.version.cuda}")
# print(
#     f"GPU device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}"
# )

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU device: NVIDIA GeForce RTX 5070


In [7]:
import hashlib
import json
import tempfile
import traceback
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path

from PIL import Image

from src.mfe.artifacts.token_count import count_tokens
from src.mfe.config.logging import create_logger
from src.mfe.converters.soffice import convert_office_to_pdf
from src.mfe.extractors.deepseek_ocr import extract_text_from_image_bytes
from src.mfe.extractors.enums.ocr_type import OCRModel
from src.mfe.extractors.ocr import ocr_pdf
from src.mfe.ingest.unzip import extract_file, is_compressed

logger, error_logger = create_logger(name="deep_seek_2_pipeline")

files_to_rename_dir = os.path.join(root_dir, "data", "input", "raw", "files_to_rename")

if not os.path.exists(files_to_rename_dir):
    raise FileNotFoundError(
        f"Diretório com dados não encontrado: {files_to_rename_dir}"
    )

output_jsonl_dir = os.path.join(
    root_dir, "data", "output", "deep_seek_2_parallel", "jsonl"
)
output_text_dir = os.path.join(
    root_dir, "data", "output", "deep_seek_2_parallel", "text_files"
)
output_images_dir = os.path.join(
    root_dir, "data", "output", "deep_seek_2_parallel", "images"
)
output_markdown_dir = os.path.join(
    root_dir, "data", "output", "deep_seek_2_parallel", "markdown"
)

for d in [output_jsonl_dir, output_text_dir, output_images_dir, output_markdown_dir]:
    os.makedirs(d, exist_ok=True)

output_jsonl_final_dir = os.path.join(
    root_dir, "data", "output", "deep_seek_2_parallel", "jsonl_final"
)

# fmt: off
TEXTUAL_EXTENSIONS = {
    ".txt", ".py", ".java", ".js", ".ts", ".jsx", ".tsx",
    ".html", ".htm", ".css", ".xml", ".json", ".yaml", ".yml",
    ".md", ".rst", ".csv", ".sh", ".bash", ".rb", ".php",
    ".c", ".cpp", ".h", ".hpp", ".go", ".rs", ".swift",
    ".kt", ".scala", ".r", ".sql",
    ".g", ".g4", ".j", "por",
}
RAW_FILE_EXTENSIONS = {".pdf"}
OFFICE_EXTENSIONS = {
    ".docx", ".doc", ".pptx", ".ppt", ".pps", ".ppsx", ".xlsx", ".xls", ".rtf",
}
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".jp", ".webp", ".pgm"}
# fmt: on

In [ ]:
BATCH_SIZE = 200


def generate_record_id(file_path: str, relative_path: str) -> str:
    """SHA-256 do conteúdo + caminho relativo — garante unicidade mesmo para
    arquivos com conteúdo idêntico em locais diferentes."""
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256.update(chunk)

    sha256.update(relative_path.encode("utf-8"))
    return sha256.hexdigest()


def save_batch(records: list, batch_index: int):
    batch_path = os.path.join(output_jsonl_dir, f"batch_{batch_index:04d}.jsonl")
    with open(batch_path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    logger.info(f"Batch {batch_index} salvo ({len(records)} registros) → {batch_path}")


def collect_all_files(base_dir: str) -> list:
    all_files = []
    for root, _dirs, files in os.walk(base_dir):
        for filename in files:
            all_files.append(os.path.join(root, filename))
    return sorted(all_files)


def image_file_to_png_bytes(file_path: str) -> bytes:
    """Abre qualquer formato de imagem suportado pelo Pillow e retorna bytes PNG."""
    img = Image.open(file_path).convert("RGB")
    buf = BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


async def process_file(
    filepath: str, relative_filepath: str, records: list, batch_index: int
) -> tuple:
    logger.info(f"\nProcessando: {filepath}")
    ext = Path(filepath).suffix.lower()

    record_id = generate_record_id(filepath, relative_filepath)
    text_output_path = os.path.join(output_text_dir, f"{record_id}.txt")
    images_output_path = os.path.join(output_images_dir, record_id)

    text_file_s3_uri = f"output/deep_seek_2/text_file/{record_id}.txt"
    images_s3_uri = f"output/deep_seek_2/images/{record_id}/"

    start_dt = datetime.now(timezone.utc)
    extraction_started = start_dt.isoformat()
    status = "FAILED"
    tokens_count = 0
    page_count = None

    try:
        if ext in IMAGE_EXTENSIONS:
            image_bytes = image_file_to_png_bytes(filepath)
            content = await extract_text_from_image_bytes(
                image_bytes=image_bytes,
                local_base_directory=images_output_path,
                upload_s3_enabled=False,
            )

            with open(text_output_path, "w", encoding="utf-8") as f:
                f.write(content)

            tokens_count = count_tokens(content)
            status = "EXTRACTED"
        elif ext in OFFICE_EXTENSIONS or ext in RAW_FILE_EXTENSIONS:
            pdf_path = filepath
            tmp_pdf = None

            if ext in OFFICE_EXTENSIONS:
                tmp_fd, tmp_pdf = tempfile.mkstemp(suffix=".pdf")
                os.close(tmp_fd)
                convert_docx_to_pdf(filepath, tmp_pdf)
                pdf_path = tmp_pdf

            try:
                text, page_count = await ocr_pdf(
                    pdf_path=pdf_path,
                    ocr_model=OCRModel.DEEPSEEK,
                    local_base_directory=images_output_path,
                    local_limit_quantity=20,
                    upload_s3_enabled=False,
                )
            finally:
                if tmp_pdf and os.path.exists(tmp_pdf):
                    os.remove(tmp_pdf)

            with open(text_output_path, "w", encoding="utf-8") as f:
                f.write(text)

            tokens_count = count_tokens(text)
            status = "EXTRACTED"
        else:
            logger.info(
                f"[EXTRACTED] Extensão sem necessidade de extração: {ext} → {relative_filepath}"
            )

            with open(filepath, "r", encoding="utf-8", errors="replace") as f:
                content = f.read()
            with open(text_output_path, "w", encoding="utf-8") as f:
                f.write(content)

            tokens_count = 0
            try:
                tokens_count = count_tokens(content)
            except Exception as e:
                error_logger.error(
                    f"[ERROR] Falha ao contar tokens em {relative_filepath}: {e}"
                )
                traceback.print_exc()

            status = "EXTRACTED"

    except Exception as e:
        error_logger.error(f"[ERROR] {relative_filepath}: {e}")
        traceback.print_exc()

    end_dt = datetime.now(timezone.utc)
    extraction_finished = end_dt.isoformat()
    extraction_duration = (end_dt - start_dt).total_seconds()

    records.append(
        {
            "id": record_id,
            "filepath": relative_filepath,
            "file_format": ext,
            "status": status,
            "tokens_count": tokens_count,
            "page_count": page_count,
            "text_file_s3_uri": text_file_s3_uri,
            "images_s3_uri": images_s3_uri,
            "extraction_started": extraction_started,
            "extraction_finished": extraction_finished,
            "extraction_duration_in_seconds": extraction_duration,
        }
    )

    if len(records) >= BATCH_SIZE:
        save_batch(records, batch_index)
        batch_index += 1
        records = []

    return records, batch_index


# ---------------------------------------------------------------------------
# Pipeline principal
# ---------------------------------------------------------------------------
all_files = collect_all_files(files_to_rename_dir)
logger.info(f"Arquivos encontrados: {len(all_files)}")

records = []
batch_index = 0

for filepath in all_files:
    if is_compressed(filepath):
        logger.info(f"\n[COMPRESSED] Extraindo: {filepath}")
        zip_relative = os.path.relpath(filepath, root_dir)

        try:
            with tempfile.TemporaryDirectory() as tmpdir:
                extract_file(filepath, tmpdir)
                for extracted_path in collect_all_files(tmpdir):
                    internal_path = os.path.relpath(extracted_path, tmpdir)
                    relative_filepath = f"{zip_relative}/{internal_path}"
                    records, batch_index = await process_file(
                        extracted_path, relative_filepath, records, batch_index
                    )
        except Exception as e:
            error_logger.error(f"[ERROR] Falha ao extrair {filepath}: {e}")
    else:
        relative_filepath = os.path.relpath(filepath, root_dir)
        records, batch_index = await process_file(
            filepath, relative_filepath, records, batch_index
        )

if records:
    save_batch(records, batch_index)

logger.info(
    f"\nPipeline concluída! {len(all_files)} arquivos processados, {batch_index + (1 if records else 0)} batches gerados."
)

# Create JSONL Batches with full content

In [ ]:
import os

from src.mfe.db.session import SessionLocal
from src.mfe.db.sqlite_metadata import upsert_record
from src.mfe.ingest.class_metadata import build_metadata_lookup, get_class_metadata

PIPELINE_NAME = "deepseek"

EXCLUDED_FIELDS = {
    "text_file_s3_uri",
    "extraction_started",
    "extraction_finished",
    "extraction_duration_in_seconds",
    "status",
    "filepath",
}

output_jsonl_final_dir = os.path.join(
    root_dir, "data", "output", "deep_seek_2", "jsonl_final"
)
os.makedirs(output_jsonl_final_dir, exist_ok=True)

db_session = SessionLocal()
logger.info("Sessão PostgreSQL pronta")

metadata_json_path = os.path.join(
    root_dir,
    "data",
    "input",
    "raw",
    "mongo-info",
    "soberania_ufpi.departamento_computacao.json",
)
file_metadata_lookup = build_metadata_lookup(metadata_json_path)
logger.info(
    f"Lookup de metadados de turmas construído: {len(file_metadata_lookup)} arquivos mapeados"
)

# ---------------------------------------------------------------------------
# Enrich JSONL records with text content and class_metadata
# ---------------------------------------------------------------------------
jsonl_files = sorted(Path(output_jsonl_dir).glob("*.jsonl"))
logger.info(f"Arquivos JSONL encontrados: {len(jsonl_files)}")

for jsonl_path in jsonl_files:
    final_records = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)

            text_path = os.path.join(output_text_dir, f"{record['id']}.txt")
            try:
                with open(text_path, "r", encoding="utf-8", errors="replace") as tf:
                    text_content = tf.read()
            except FileNotFoundError:
                logger.warning(f"Arquivo de texto não encontrado: {text_path}")
                text_content = None

            record["class_metadata"] = get_class_metadata(
                record.get("filepath", ""), file_metadata_lookup
            )

            upsert_record(db_session, record, PIPELINE_NAME)

            new_record = {k: v for k, v in record.items() if k not in EXCLUDED_FIELDS}
            new_record["text_content"] = text_content
            final_records.append(new_record)

    output_path = os.path.join(output_jsonl_final_dir, jsonl_path.name)
    with open(output_path, "w", encoding="utf-8") as f:
        for record in final_records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    logger.info(f"{jsonl_path.name} → {len(final_records)} registros → {output_path}")

db_session.close()
logger.info(
    f"Concluído! {len(jsonl_files)} arquivos gerados em {output_jsonl_final_dir}"
)


2026-03-15 18:09:18,334 INFO:deep_seek_2_pipeline:Banco SQLite pronto: /home/victor-wsl/programming/curso-ia/data/output/deep_seek_2/database/extraction_records.db
2026-03-15 18:09:18,376 INFO:deep_seek_2_pipeline:Lookup de metadados de turmas construído: 1012 arquivos mapeados
2026-03-15 18:09:18,378 INFO:deep_seek_2_pipeline:Arquivos JSONL encontrados: 1
2026-03-15 18:09:18,391 INFO:deep_seek_2_pipeline:batch_0000.jsonl → 1 registros → /home/victor-wsl/programming/curso-ia/data/output/deep_seek_2/jsonl_final/batch_0000.jsonl
2026-03-15 18:09:18,392 INFO:deep_seek_2_pipeline:Concluído! 1 arquivos gerados em /home/victor-wsl/programming/curso-ia/data/output/deep_seek_2/jsonl_final


# Upload Dataset to Hugging Face Hub

In [9]:
from huggingface_hub import CommitOperationAdd, CommitOperationDelete, HfApi

HF_TOKEN = os.getenv("HF_TOKEN")  # token em .env com permissão write
HF_USERNAME = os.getenv("HF_USERNAME")  # seu username no HF
DATASET_NAME = "ia-course"  # nome do repositório de dataset
DATASET_VERSION = "0.0.4"
EXTRACTION_MODEL = "deepseek-ocr-2"

repo_id = f"Cursos-IA/{DATASET_NAME}"

# ---------------------------------------------------------------------------
# Dataset card (README.md)
# ---------------------------------------------------------------------------
dataset_card = f"""\
---
license: other
tags:
  - ocr
  - pdf
  - text-extraction
  - {EXTRACTION_MODEL}
version: "{DATASET_VERSION}"
configs:
  - config_name: default
    data_files:
      - split: train
        path: "data/*.jsonl"
---

# {DATASET_NAME}

Extracted text dataset generated from PDF and document files using OCR. This version has been processed with the {EXTRACTION_MODEL} model, and was merged with the metadata available in the mongodb.
In this version was added files inside of zip files, so now we have more records than the previous version (v0.0.2) that was generated with the same extraction model but without support for zip files.

## Extraction details

| Field | Value |
|---|---|
| Extraction model | `{EXTRACTION_MODEL}` |
| Dataset version | `{DATASET_VERSION}` |
| Pipeline | DeepSeek OCR Pipeline |

## Schema

Each record contains:

| Field | Description |
|---|---|
| `id` | SHA-256 hash of file content + relative path |
| `file_format` | File extension (`.pdf`, `.docx`, etc.) |
| `tokens_count` | Token count (GPT-4o tokenizer) |
| `images_s3_uri` | S3 URI for extracted page images |
| `text_content` | Full extracted text content |
| `class_metadata` | Metadata about the class (turma/plano_aula) if matched |


## Versions

| Version | Tag | Notes |
|---|---|---|
| 0.0.2 | `v0.0.2` | First extraction using `{EXTRACTION_MODEL}` |
| 0.0.3 | `v0.0.3` | Second extraction using `{EXTRACTION_MODEL}` with zip support |
| 0.0.4 | `v0.0.4` | Third extraction using `{EXTRACTION_MODEL}` with zip support, xml conversion and audio and video extraction |
"""

api = HfApi(token=HF_TOKEN)

api.create_repo(
    repo_id=repo_id,
    repo_type="dataset",
    exist_ok=True,
    private=True,
)
logger.info(f"Repositório pronto: https://huggingface.co/datasets/{repo_id}")

# ---------------------------------------------------------------------------
# Delete existing data files from previous version
# ---------------------------------------------------------------------------
existing_files = [
    f.rfilename
    for f in api.list_repo_tree(repo_id=repo_id, repo_type="dataset", recursive=True)
    if hasattr(f, "rfilename") and f.rfilename.startswith("data/")
]
logger.info(f"Arquivos existentes para deletar: {len(existing_files)}")

operations = [CommitOperationDelete(path_in_repo=path) for path in existing_files]

# ---------------------------------------------------------------------------
# Add new JSONL files + README
# ---------------------------------------------------------------------------
jsonl_final_dir = Path(output_jsonl_final_dir)
operations += [
    CommitOperationAdd(
        path_in_repo=f"data/{jsonl_file.name}",
        path_or_fileobj=str(jsonl_file),
    )
    for jsonl_file in sorted(jsonl_final_dir.glob("*.jsonl"))
]

operations.append(
    CommitOperationAdd(
        path_in_repo="README.md",
        path_or_fileobj=dataset_card.encode("utf-8"),
    )
)

logger.info(
    f"Fazendo upload de {len([o for o in operations if isinstance(o, CommitOperationAdd)]) - 1} arquivos JSONL + README ({len(existing_files)} arquivos antigos deletados)..."
)

commit_info = api.create_commit(
    repo_id=repo_id,
    repo_type="dataset",
    operations=operations,
    commit_message=f"feat: add extraction v{DATASET_VERSION} using {EXTRACTION_MODEL}",
)
logger.info(f"Commit: {commit_info.commit_url}")

# Cria tag git para marcar a versão imutável
api.create_tag(
    repo_id=repo_id,
    repo_type="dataset",
    tag=f"v{DATASET_VERSION}",
    tag_message=f"First extraction using {EXTRACTION_MODEL}",
    revision=commit_info.oid,
)
logger.info(
    f"Tag v{DATASET_VERSION} criada → https://huggingface.co/datasets/{repo_id}/tree/v{DATASET_VERSION}"
)


2026-03-20 10:52:48,570 INFO:deep_seek_2_pipeline:Repositório pronto: https://huggingface.co/datasets/Cursos-IA/ia-course
2026-03-20 10:52:48,972 INFO:deep_seek_2_pipeline:Arquivos existentes para deletar: 10
2026-03-20 10:52:49,012 INFO:deep_seek_2_pipeline:Fazendo upload de 8 arquivos JSONL + README (10 arquivos antigos deletados)...
Processing Files (6 / 6): 100%|██████████| 44.8MB / 44.8MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
2026-03-20 10:52:52,794 INFO:deep_seek_2_pipeline:Commit: https://huggingface.co/datasets/Cursos-IA/ia-course/commit/31026fa1cb6df4219a87f813f03bbd19c473dd7f
2026-03-20 10:52:53,492 INFO:deep_seek_2_pipeline:Tag v0.0.4 criada → https://huggingface.co/datasets/Cursos-IA/ia-course/tree/v0.0.4


In [ ]:
# # ---------------------------------------------------------------------------
# # Delete version 0.0.4 from Hugging Face Hub
# # ---------------------------------------------------------------------------
# from huggingface_hub import CommitOperationDelete

# VERSION_TO_DELETE = "0.0.4"

# # Delete the git tag
# api.delete_tag(
#     repo_id=repo_id,
#     repo_type="dataset",
#     tag=f"v{VERSION_TO_DELETE}",
# )
# logger.info(f"Tag v{VERSION_TO_DELETE} deletada")

# # Delete all data files uploaded in that version
# files_to_delete = [f"data/{f.name}" for f in sorted(jsonl_final_dir.glob("*.jsonl"))]
# files_to_delete.append("README.md")

# delete_operations = [
#     CommitOperationDelete(path_in_repo=path) for path in files_to_delete
# ]

# commit_info = api.create_commit(
#     repo_id=repo_id,
#     repo_type="dataset",
#     operations=delete_operations,
#     commit_message=f"revert: remove wrong upload v{VERSION_TO_DELETE}",
# )
# logger.info(f"Arquivos removidos. Commit: {commit_info.commit_url}")


2026-03-20 10:52:44,379 INFO:deep_seek_2_pipeline:Tag v0.0.4 deletada
2026-03-20 10:52:44,905 INFO:deep_seek_2_pipeline:Arquivos removidos. Commit: https://huggingface.co/datasets/Cursos-IA/ia-course/commit/90de8f6218231cbcdc8e47e7195960357a89af4f
